# Advanced Python Set Operations — Problems with Complete Solutions

This notebook develops practical mastery of Python sets through progressively harder examples and **20 advanced problems with solutions**.

## Learning goals

By the end, you should be able to:

- choose correctly among union, intersection, difference, and symmetric difference;
- reason about subset/superset relationships and partial ordering;
- use method forms versus operator forms safely;
- apply `set`, `frozenset`, dictionary views, and set comprehensions;
- design efficient solutions for data reconciliation, permissions, graphs, search, similarity, and constraint problems;
- avoid common correctness and performance mistakes;
- test set-based code without depending on arbitrary iteration order.

> **Best practice:** Treat sets as unordered. Sort only when displaying deterministic output.

## 0. Notebook conventions

The notebook uses:

- type hints and docstrings for reusable functions;
- assertions as executable specifications;
- deterministic display through `sorted(...)`;
- fresh copies when mutation would otherwise surprise the caller;
- small, dependency-free examples that run with the Python standard library.

In [1]:
from collections import Counter, defaultdict
from collections.abc import Hashable, Iterable, Mapping, Sequence
from functools import reduce
from itertools import combinations
from typing import TypeVar

T = TypeVar("T", bound=Hashable)

def show(values: Iterable[T]) -> list[T]:
    """Return a deterministic, sorted representation for display/tests."""
    return sorted(values)

# 1. Compact but deeper recap

## 1.1 Core binary operations

For sets `A` and `B`:

- `A & B`: values in both;
- `A | B`: values in either;
- `A - B`: values in `A` but not `B`;
- `A ^ B`: values in exactly one of the two;
- `A <= B`: subset, allowing equality;
- `A < B`: proper subset;
- `A.isdisjoint(B)`: no shared elements.

In [2]:
A = {1, 2, 3, 4}
B = {3, 4, 5, 6}

results = {
    "intersection": show(A & B),
    "union": show(A | B),
    "A_minus_B": show(A - B),
    "B_minus_A": show(B - A),
    "symmetric_difference": show(A ^ B),
    "A_subset_B": A <= B,
    "disjoint": A.isdisjoint(B),
}
results

{'intersection': [3, 4],
 'union': [1, 2, 3, 4, 5, 6],
 'A_minus_B': [1, 2],
 'B_minus_A': [5, 6],
 'symmetric_difference': [1, 2, 5, 6],
 'A_subset_B': False,
 'disjoint': False}

## 1.2 Operators require set-like operands; methods accept general iterables

This distinction is useful when one side is a list, tuple, generator, dictionary, or dictionary view.

In [3]:
left = {1, 2, 3}

method_result = left.intersection([2, 3, 3, 4])
operator_result = left & set([2, 3, 3, 4])

assert method_result == operator_result == {2, 3}
show(method_result)

[2, 3]

In [4]:
required_fields = {"id", "email"}
record = {"id": 7, "email": "a@example.com", "name": "Ada"}

present_required = required_fields.intersection(record)
missing_required = required_fields.difference(record)

show(present_required), show(missing_required)

(['email', 'id'], [])

## 1.3 Non-mutating versus in-place operations

Prefer non-mutating expressions when preserving the input matters. Use update methods when mutation is intentional and documented.

In [5]:
base = {1, 2, 3}
other = {3, 4}

new_union = base | other
assert base == {1, 2, 3}

mutable = base.copy()
mutable.update(other)
mutable.intersection_update({2, 3, 4})
mutable.difference_update({4})
mutable.symmetric_difference_update({1, 5})

show(new_union), show(mutable)

([1, 2, 3, 4], [1, 2, 3, 5])

## 1.4 Hashability and `frozenset`

Set elements must be hashable. A mutable `set` is unhashable, but a `frozenset` is hashable and can be nested inside another set or used as a dictionary key.

In [6]:
teams = {
    frozenset({"Ana", "Bo"}),
    frozenset({"Cy", "Di"}),
}

pair_scores = {
    frozenset({"Ana", "Bo"}): 9,
    frozenset({"Cy", "Di"}): 7,
}

pair_scores[frozenset({"Bo", "Ana"})], teams

(9, {frozenset({'Ana', 'Bo'}), frozenset({'Cy', 'Di'})})

## 1.5 Set comprehensions

In [7]:
numbers = range(-10, 11)
squares_of_even_magnitudes = {abs(n) ** 2 for n in numbers if n % 2 == 0}
show(squares_of_even_magnitudes)

[0, 4, 16, 36, 64, 100]

## 1.6 Complexity intuition

Average-case behavior for hash-based sets:

| Operation | Typical cost |
|---|---:|
| `x in s` | `O(1)` |
| `s.add(x)` / `s.discard(x)` | `O(1)` |
| `A & B` | proportional to the smaller set |
| `A | B`, `A ^ B` | proportional to combined input size |
| `A - B` | proportional to the left operand |

These are average-case guidelines, not hard real-time guarantees.

# 2. Advanced worked examples

## Example 1 — Reconcile records from multiple systems

We want values that are:

1. in all systems;
2. in at least one system;
3. unique to each system;
4. present in an odd number of systems.

In [8]:
crm = {"A01", "A02", "A03", "A05"}
billing = {"A02", "A03", "A04", "A05"}
support = {"A03", "A05", "A06"}

all_three = crm & billing & support
any_system = crm | billing | support

unique_to_crm = crm - billing - support
unique_to_billing = billing - crm - support
unique_to_support = support - crm - billing

# Chained XOR means odd parity, not necessarily "exactly one" for 3+ sets.
odd_parity = crm ^ billing ^ support

{
    "all_three": show(all_three),
    "any_system": show(any_system),
    "unique_to_crm": show(unique_to_crm),
    "unique_to_billing": show(unique_to_billing),
    "unique_to_support": show(unique_to_support),
    "odd_parity": show(odd_parity),
}

{'all_three': ['A03', 'A05'],
 'any_system': ['A01', 'A02', 'A03', 'A04', 'A05', 'A06'],
 'unique_to_crm': ['A01'],
 'unique_to_billing': ['A04'],
 'unique_to_support': ['A06'],
 'odd_parity': ['A01', 'A03', 'A04', 'A05', 'A06']}

### Exactly one versus XOR parity

For three or more sets, chained XOR contains elements appearing an **odd** number of times. An element in all three sets appears in the result, so use counts for “exactly one.”

In [9]:
def elements_with_frequency(sets: Iterable[set[T]], frequency: int) -> set[T]:
    """Return elements occurring in exactly `frequency` input sets."""
    counts = Counter(item for current in sets for item in current)
    return {item for item, count in counts.items() if count == frequency}

exactly_one = elements_with_frequency([crm, billing, support], 1)
exactly_two = elements_with_frequency([crm, billing, support], 2)
exactly_three = elements_with_frequency([crm, billing, support], 3)

show(exactly_one), show(exactly_two), show(exactly_three)

(['A01', 'A04', 'A06'], ['A02'], ['A03', 'A05'])

## Example 2 — Role-based access control

In [10]:
ROLE_PERMISSIONS = {
    "viewer": {"read"},
    "editor": {"read", "write", "comment"},
    "auditor": {"read", "export", "view_logs"},
    "admin": {"read", "write", "comment", "delete", "export", "view_logs"},
}

def effective_permissions(
    roles: Iterable[str],
    *,
    denied: Iterable[str] = (),
) -> set[str]:
    """Combine role permissions, then remove explicitly denied actions."""
    granted: set[str] = set()
    for role in roles:
        try:
            granted.update(ROLE_PERMISSIONS[role])
        except KeyError as exc:
            raise ValueError(f"Unknown role: {role!r}") from exc
    return granted.difference(denied)

permissions = effective_permissions(
    ["editor", "auditor"],
    denied={"export"},
)

required_for_publish = {"read", "write"}
can_publish = required_for_publish <= permissions

show(permissions), can_publish

(['comment', 'read', 'view_logs', 'write'], True)

## Example 3 — Jaccard similarity for duplicate detection

In [11]:
def jaccard(left: Iterable[T], right: Iterable[T]) -> float:
    """Compute Jaccard similarity; define two empty sets as perfectly similar."""
    a, b = set(left), set(right)
    union = a | b
    return 1.0 if not union else len(a & b) / len(union)

doc_a = "python sets are fast for membership tests".split()
doc_b = "sets provide fast membership testing in python".split()

round(jaccard(doc_a, doc_b), 3)

0.4

In [12]:
def shingles(text: str, size: int = 3) -> set[tuple[str, ...]]:
    """Return contiguous word shingles as hashable tuples."""
    if size <= 0:
        raise ValueError("size must be positive")
    tokens = text.lower().split()
    return {
        tuple(tokens[i : i + size])
        for i in range(len(tokens) - size + 1)
    }

text_1 = "sets make membership tests concise and fast"
text_2 = "python sets make membership tests very concise and fast"

round(jaccard(shingles(text_1), shingles(text_2)), 3)

0.333

## Example 4 — Boolean search with an inverted index

In [13]:
documents = {
    1: "python sets support fast membership",
    2: "python dictionaries store key value pairs",
    3: "set algebra supports union and intersection",
    4: "fast search uses an inverted index",
}

def build_inverted_index(docs: Mapping[int, str]) -> dict[str, set[int]]:
    index: dict[str, set[int]] = defaultdict(set)
    for doc_id, text in docs.items():
        for token in set(text.lower().split()):
            index[token].add(doc_id)
    return dict(index)

index = build_inverted_index(documents)

python_and_sets = index.get("python", set()) & index.get("sets", set())
union_or_intersection = index.get("union", set()) | index.get("intersection", set())
fast_but_not_search = index.get("fast", set()) - index.get("search", set())

show(python_and_sets), show(union_or_intersection), show(fast_but_not_search)

([1], [3], [1])

## Example 5 — Graph analytics with adjacency sets

In [14]:
graph: dict[str, set[str]] = {
    "A": {"B", "C", "D"},
    "B": {"A", "C", "E"},
    "C": {"A", "B", "D", "E"},
    "D": {"A", "C"},
    "E": {"B", "C"},
}

def common_neighbors(graph: Mapping[T, set[T]], u: T, v: T) -> set[T]:
    return graph[u] & graph[v]

def friend_recommendations(graph: Mapping[T, set[T]], user: T) -> list[tuple[T, int]]:
    """Rank non-neighbors by number of mutual neighbors."""
    existing = graph[user] | {user}
    candidates = set(graph) - existing
    ranked = [
        (candidate, len(graph[user] & graph[candidate]))
        for candidate in candidates
    ]
    return sorted(ranked, key=lambda pair: (-pair[1], pair[0]))

show(common_neighbors(graph, "A", "E")), friend_recommendations(graph, "A")

(['B', 'C'], [('E', 2)])

In [15]:
def count_triangles(graph: Mapping[T, set[T]]) -> int:
    """Count undirected triangles once each."""
    nodes = sorted(graph)
    count = 0
    for u, v in combinations(nodes, 2):
        if v in graph[u]:
            count += len(graph[u] & graph[v])
    return count // 3

count_triangles(graph)

3

## Example 6 — Candidate elimination in a mini Sudoku row

In [16]:
DIGITS = set(range(1, 10))

def sudoku_candidates(
    row_values: Iterable[int],
    column_values: Iterable[int],
    box_values: Iterable[int],
) -> set[int]:
    """Return legal values for one Sudoku cell; zero denotes blank."""
    used = (set(row_values) | set(column_values) | set(box_values)) - {0}
    return DIGITS - used

candidates = sudoku_candidates(
    row_values=[5, 3, 0, 6, 7, 8, 9, 1, 2],
    column_values=[0, 6, 0, 8, 4, 7, 0, 0, 0],
    box_values=[5, 3, 0, 6, 0, 0, 0, 9, 8],
)
show(candidates)

[]

## Example 7 — Greedy set cover

In [17]:
def greedy_set_cover(
    universe: set[T],
    candidates: Mapping[str, set[T]],
) -> list[str]:
    """Return a greedy cover or raise if coverage is impossible."""
    uncovered = set(universe)
    chosen: list[str] = []

    while uncovered:
        name, covered = max(
            candidates.items(),
            key=lambda item: len(item[1] & uncovered),
        )
        newly_covered = covered & uncovered
        if not newly_covered:
            raise ValueError(f"Cannot cover: {show(uncovered)}")
        chosen.append(name)
        uncovered.difference_update(newly_covered)

    return chosen

universe = {"a", "b", "c", "d", "e", "f"}
stations = {
    "S1": {"a", "b", "c"},
    "S2": {"b", "d"},
    "S3": {"c", "e"},
    "S4": {"d", "e", "f"},
}

greedy_set_cover(universe, stations)

['S1', 'S4']

# 3. Advanced problems with solutions

Each problem is followed by a complete reference solution and executable checks.

## Problem 1 — Multi-source inventory reconciliation

Three warehouses report product IDs. Write a function returning:

- products reported by every warehouse;
- products reported by at least one warehouse;
- products appearing in exactly one warehouse;
- products appearing in exactly two warehouses;
- products whose presence has odd parity.

The function must support any number of warehouses, including zero.

In [18]:
warehouse_sets = [
    {"P1", "P2", "P3", "P7"},
    {"P2", "P3", "P4", "P7"},
    {"P3", "P5", "P7"},
]

### Solution 1

In [19]:
def inventory_summary(warehouses: Sequence[set[T]]) -> dict[str, set[T]]:
    """Summarize membership across an arbitrary number of warehouses."""
    if not warehouses:
        return {
            "all": set(),
            "any": set(),
            "exactly_one": set(),
            "exactly_two": set(),
            "odd_parity": set(),
        }

    counts = Counter(item for warehouse in warehouses for item in warehouse)
    all_warehouses = set.intersection(*(set(w) for w in warehouses))
    any_warehouse = set.union(*(set(w) for w in warehouses))
    odd_parity = reduce(set.symmetric_difference, warehouses, set())

    return {
        "all": all_warehouses,
        "any": any_warehouse,
        "exactly_one": {x for x, n in counts.items() if n == 1},
        "exactly_two": {x for x, n in counts.items() if n == 2},
        "odd_parity": odd_parity,
    }

summary = inventory_summary(warehouse_sets)

assert summary["all"] == {"P3", "P7"}
assert summary["exactly_one"] == {"P1", "P4", "P5"}
assert summary["exactly_two"] == {"P2"}
assert summary["odd_parity"] == {"P1", "P3", "P4", "P5", "P7"}

{k: show(v) for k, v in summary.items()}

{'all': ['P3', 'P7'],
 'any': ['P1', 'P2', 'P3', 'P4', 'P5', 'P7'],
 'exactly_one': ['P1', 'P4', 'P5'],
 'exactly_two': ['P2'],
 'odd_parity': ['P1', 'P3', 'P4', 'P5', 'P7']}

## Problem 2 — Permission audit with explicit deny

A user receives permissions from several roles. Explicit denies override grants.

Return effective permissions, missing permissions, excess permissions, exact-match status, and authorization status.

In [20]:
roles = {
    "author": {"read", "write", "submit"},
    "reviewer": {"read", "comment", "approve"},
    "publisher": {"read", "publish"},
}
user_roles = {"author", "reviewer"}
denied = {"approve"}
operation_requirements = {
    "draft": {"read", "write"},
    "submit": {"read", "write", "submit"},
    "review": {"read", "comment"},
}

### Solution 2

In [21]:
def audit_permissions(
    role_permissions: Mapping[str, set[str]],
    assigned_roles: Iterable[str],
    denied: Iterable[str],
    operation_requirements: Mapping[str, set[str]],
    target_operation: str,
) -> dict[str, object]:
    grants: set[str] = set()
    for role in assigned_roles:
        if role not in role_permissions:
            raise ValueError(f"Unknown role: {role}")
        grants.update(role_permissions[role])

    effective = grants - set(denied)
    required = operation_requirements[target_operation]
    all_declared = set().union(*operation_requirements.values())

    return {
        "effective": effective,
        "missing": required - effective,
        "excess": effective - all_declared,
        "exact_match": effective == required,
        "authorized": required <= effective,
    }

audit = audit_permissions(
    roles,
    user_roles,
    denied,
    operation_requirements,
    "submit",
)

assert audit["effective"] == {"read", "write", "submit", "comment"}
assert audit["missing"] == set()
assert audit["authorized"] is True
audit

{'effective': {'comment', 'read', 'submit', 'write'},
 'missing': set(),
 'excess': set(),
 'exact_match': False,
 'authorized': True}

## Problem 3 — Cohort retention analysis

Compute retention, churn, resurrection, and brand-new users across three months.

In [22]:
active = {
    "Jan": {"u1", "u2", "u3", "u4"},
    "Feb": {"u2", "u3", "u5"},
    "Mar": {"u1", "u2", "u5", "u6"},
}

### Solution 3

In [23]:
def retention_rate(previous: set[T], current: set[T]) -> float:
    """Fraction of previous users retained; empty previous cohort returns 1.0."""
    return 1.0 if not previous else len(previous & current) / len(previous)

jan, feb, mar = active["Jan"], active["Feb"], active["Mar"]

retained_jan_to_feb = jan & feb
churned_after_jan = jan - feb
resurrected_in_mar = (jan - feb) & mar
brand_new_in_mar = mar - jan - feb

cohort_result = {
    "retention_rate": retention_rate(jan, feb),
    "retained": show(retained_jan_to_feb),
    "churned": show(churned_after_jan),
    "resurrected": show(resurrected_in_mar),
    "brand_new": show(brand_new_in_mar),
}

assert cohort_result["retention_rate"] == 0.5
assert resurrected_in_mar == {"u1"}
assert brand_new_in_mar == {"u6"}
cohort_result

{'retention_rate': 0.5,
 'retained': ['u2', 'u3'],
 'churned': ['u1', 'u4'],
 'resurrected': ['u1'],
 'brand_new': ['u6']}

## Problem 4 — Near-duplicate detection using word shingles

Compare all document pairs with Jaccard similarity and return pairs meeting a threshold.

In [24]:
corpus = {
    "d1": "python sets make membership testing fast and expressive",
    "d2": "python sets make membership checks fast and expressive",
    "d3": "sql joins combine rows from related tables",
    "d4": "membership testing with python sets is fast and expressive",
}

### Solution 4

In [25]:
def near_duplicate_pairs(
    docs: Mapping[str, str],
    *,
    shingle_size: int = 2,
    threshold: float = 0.4,
) -> list[tuple[str, str, float]]:
    if shingle_size <= 0:
        raise ValueError("shingle_size must be positive")
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be between 0 and 1")

    features = {
        doc_id: shingles(text, shingle_size)
        for doc_id, text in docs.items()
    }

    matches: list[tuple[str, str, float]] = []
    for left, right in combinations(sorted(docs), 2):
        score = jaccard(features[left], features[right])
        if score >= threshold:
            matches.append((left, right, round(score, 3)))
    return matches

near_duplicate_pairs(corpus, shingle_size=2, threshold=0.3)

[('d1', 'd2', 0.556), ('d1', 'd4', 0.364)]

## Problem 5 — Elements appearing in at least `k` sets

Support arbitrary iterables and count membership per distinct group, not duplicate occurrences inside one group.

### Solution 5

In [26]:
def at_least_k(
    groups: Iterable[Iterable[T]],
    k: int,
) -> set[T]:
    """Return items occurring in at least k distinct groups."""
    materialized = [set(group) for group in groups]

    if k <= 0:
        raise ValueError("k must be positive")
    if not materialized or k > len(materialized):
        return set()

    counts = Counter(item for group in materialized for item in group)
    return {item for item, count in counts.items() if count >= k}

groups = [
    [1, 1, 2, 3],
    [2, 3, 4],
    [3, 4, 5],
    [3, 6],
]

assert at_least_k(groups, 2) == {2, 3, 4}
assert at_least_k(groups, 3) == {3}
show(at_least_k(groups, 2))

[2, 3, 4]

## Problem 6 — Validate dictionary schemas

Report missing, unexpected, shared, and total keys.

In [27]:
records = [
    {"id": 1, "email": "a@example.com", "name": "Ada"},
    {"id": 2, "email": "b@example.com", "name": "Bo", "phone": "123"},
    {"id": 3, "name": "Cy"},
]
required = {"id", "email"}
allowed = {"id", "email", "name", "phone"}

### Solution 6

In [28]:
def validate_records(
    records: Sequence[Mapping[str, object]],
    required: set[str],
    allowed: set[str],
) -> dict[str, object]:
    if not required <= allowed:
        raise ValueError("required keys must be a subset of allowed keys")

    per_record = []
    key_sets = [set(record) for record in records]

    for index, keys in enumerate(key_sets):
        missing = required - keys
        unexpected = keys - allowed
        per_record.append({
            "index": index,
            "missing": missing,
            "unexpected": unexpected,
            "valid": not missing and not unexpected,
        })

    shared = set.intersection(*key_sets) if key_sets else set()
    any_keys = set.union(*key_sets) if key_sets else set()

    return {
        "records": per_record,
        "shared_keys": shared,
        "any_keys": any_keys,
    }

validation = validate_records(records, required, allowed)
assert validation["shared_keys"] == {"id", "name"}
assert validation["records"][2]["missing"] == {"email"}
validation

{'records': [{'index': 0,
   'missing': set(),
   'unexpected': set(),
   'valid': True},
  {'index': 1, 'missing': set(), 'unexpected': set(), 'valid': True},
  {'index': 2, 'missing': {'email'}, 'unexpected': set(), 'valid': False}],
 'shared_keys': {'id', 'name'},
 'any_keys': {'email', 'id', 'name', 'phone'}}

## Problem 7 — Boolean query engine

Evaluate nested `TERM`, `AND`, `OR`, and `NOT` tuple expressions over an inverted index.

### Solution 7

In [29]:
Expression = tuple

def evaluate_query(
    expression: Expression,
    index: Mapping[str, set[int]],
    universe: set[int],
) -> set[int]:
    op = expression[0]

    if op == "TERM":
        return set(index.get(expression[1].lower(), set()))

    if op == "NOT":
        return universe - evaluate_query(expression[1], index, universe)

    if op in {"AND", "OR"}:
        left = evaluate_query(expression[1], index, universe)
        right = evaluate_query(expression[2], index, universe)
        return left & right if op == "AND" else left | right

    raise ValueError(f"Unknown operation: {op}")

query = (
    "AND",
    ("TERM", "python"),
    ("NOT", ("TERM", "dictionaries")),
)

query_result = evaluate_query(query, index, set(documents))
assert query_result == {1}
show(query_result)

[1]

## Problem 8 — Friend recommendations with exclusion rules

Exclude direct friends, the target user, users blocked by the target, and users who blocked the target.

In [30]:
social_graph = {
    "A": {"B", "C"},
    "B": {"A", "C", "D", "E"},
    "C": {"A", "B", "D"},
    "D": {"B", "C", "E"},
    "E": {"B", "D", "F"},
    "F": {"E"},
}
blocked = {
    "A": {"E"},
    "D": {"A"},
}

### Solution 8

In [31]:
def recommend_friends(
    graph: Mapping[T, set[T]],
    blocked: Mapping[T, set[T]],
    user: T,
) -> list[tuple[T, int]]:
    direct = graph[user]
    blocked_by_user = blocked.get(user, set())
    users_blocking_target = {
        other
        for other, blocked_users in blocked.items()
        if user in blocked_users
    }

    excluded = direct | {user} | blocked_by_user | users_blocking_target
    candidates = set(graph) - excluded

    scored = []
    for candidate in candidates:
        mutual_count = len(direct & graph[candidate])
        if mutual_count:
            scored.append((candidate, mutual_count))

    return sorted(scored, key=lambda pair: (-pair[1], pair[0]))

recommendations = recommend_friends(social_graph, blocked, "A")
assert recommendations == []

# Removing the inbound block makes D a valid recommendation with two mutual friends.
recommendations_without_inbound_block = recommend_friends(
    social_graph,
    {"A": {"E"}},
    "A",
)
assert recommendations_without_inbound_block == [("D", 2)]

recommendations, recommendations_without_inbound_block

([], [('D', 2)])

## Problem 9 — Count triangles and validate an undirected graph

Reject unknown neighbors, self-loops, and asymmetric edges before finding triangles.

### Solution 9

In [32]:
def validate_undirected_graph(graph: Mapping[T, set[T]]) -> None:
    nodes = set(graph)

    for node, neighbors in graph.items():
        unknown = neighbors - nodes
        if unknown:
            raise ValueError(f"{node!r} has unknown neighbors: {unknown}")
        if node in neighbors:
            raise ValueError(f"Self-loop at {node!r}")
        for neighbor in neighbors:
            if node not in graph[neighbor]:
                raise ValueError(f"Asymmetric edge: {node!r}-{neighbor!r}")

def triangle_nodes(graph: Mapping[T, set[T]]) -> set[frozenset[T]]:
    validate_undirected_graph(graph)
    triangles: set[frozenset[T]] = set()

    for u in graph:
        for v in graph[u]:
            for w in graph[u] & graph[v]:
                triangles.add(frozenset({u, v, w}))

    return triangles

triangles = triangle_nodes(graph)
assert len(triangles) == 3
sorted([show(triangle) for triangle in triangles])

[['A', 'B', 'C'], ['A', 'C', 'D'], ['B', 'C', 'E']]

## Problem 10 — Dependency readiness

A task is ready when every direct dependency belongs to the completed set.

In [33]:
dependencies = {
    "design": set(),
    "backend": {"design"},
    "frontend": {"design"},
    "integration": {"backend", "frontend"},
    "release": {"integration", "security"},
    "security": {"backend"},
}
completed = {"design", "backend"}

### Solution 10

In [34]:
def dependency_status(
    dependencies: Mapping[T, set[T]],
    completed: set[T],
) -> tuple[set[T], dict[T, set[T]]]:
    tasks = set(dependencies)
    unknown = set().union(*dependencies.values()) - tasks if tasks else set()

    if unknown:
        raise ValueError(f"Unknown dependencies: {unknown}")

    incomplete = tasks - completed
    ready = {
        task
        for task in incomplete
        if dependencies[task] <= completed
    }
    blocked = {
        task: dependencies[task] - completed
        for task in incomplete - ready
    }
    return ready, blocked

ready, blocked_tasks = dependency_status(dependencies, completed)

assert ready == {"frontend", "security"}
assert blocked_tasks["integration"] == {"frontend"}
show(ready), {task: show(missing) for task, missing in blocked_tasks.items()}

(['frontend', 'security'],
 {'integration': ['frontend'], 'release': ['integration', 'security']})

## Problem 11 — Schedule conflict detection

Find every pair of simultaneous meetings sharing attendees.

In [35]:
meetings = {
    "M1": {"Ana", "Bo", "Cy"},
    "M2": {"Di", "Eli"},
    "M3": {"Cy", "Fay"},
    "M4": {"Ana", "Eli"},
}

### Solution 11

In [36]:
def meeting_conflicts(
    meetings: Mapping[str, set[T]],
) -> list[tuple[str, str, set[T]]]:
    conflicts = []
    for left, right in combinations(sorted(meetings), 2):
        overlap = meetings[left] & meetings[right]
        if overlap:
            conflicts.append((left, right, overlap))
    return conflicts

conflicts = meeting_conflicts(meetings)
assert len(conflicts) == 3
[(a, b, show(overlap)) for a, b, overlap in conflicts]

[('M1', 'M3', ['Cy']), ('M1', 'M4', ['Ana']), ('M2', 'M4', ['Eli'])]

## Problem 12 — Maximal non-conflicting meeting selection

Use a deterministic greedy rule: smallest attendee count first, then meeting name.

### Solution 12

In [37]:
def greedy_disjoint_meetings(
    meetings: Mapping[str, set[T]],
) -> list[str]:
    chosen: list[str] = []
    occupied: set[T] = set()

    ordered = sorted(meetings, key=lambda name: (len(meetings[name]), name))
    for name in ordered:
        if meetings[name].isdisjoint(occupied):
            chosen.append(name)
            occupied.update(meetings[name])

    return chosen

chosen_meetings = greedy_disjoint_meetings(meetings)

for left, right in combinations(chosen_meetings, 2):
    assert meetings[left].isdisjoint(meetings[right])

chosen_meetings

['M2', 'M3']

## Problem 13 — Minimal unique tags

Find tags exclusive to each item and tags shared by every item.

In [38]:
item_tags = {
    "article": {"python", "sets", "tutorial", "beginner"},
    "video": {"python", "sets", "visual"},
    "quiz": {"python", "sets", "assessment"},
    "cheatsheet": {"python", "sets", "reference"},
}

### Solution 13

In [39]:
def tag_analysis(
    item_tags: Mapping[str, set[T]],
) -> tuple[dict[str, set[T]], set[T]]:
    names = list(item_tags)
    if not names:
        return {}, set()

    shared = set.intersection(*(item_tags[name] for name in names))
    unique_by_item: dict[str, set[T]] = {}

    for name in names:
        all_others = set().union(
            *(item_tags[other] for other in names if other != name)
        )
        unique_by_item[name] = item_tags[name] - all_others

    return unique_by_item, shared

unique_tags, shared_tags = tag_analysis(item_tags)

assert shared_tags == {"python", "sets"}
assert unique_tags["video"] == {"visual"}

{k: show(v) for k, v in unique_tags.items()}, show(shared_tags)

({'article': ['beginner', 'tutorial'],
  'video': ['visual'],
  'quiz': ['assessment'],
  'cheatsheet': ['reference']},
 ['python', 'sets'])

## Problem 14 — Powerset and constraint filtering

Return subsets as `frozenset` objects and filter valid feature bundles.

### Solution 14

In [40]:
def powerset(values: Iterable[T]) -> set[frozenset[T]]:
    """Return every subset as a frozenset."""
    items = tuple(dict.fromkeys(values))
    return {
        frozenset(combo)
        for size in range(len(items) + 1)
        for combo in combinations(items, size)
    }

features = {"search", "export", "offline", "realtime"}

valid_bundles = {
    bundle
    for bundle in powerset(features)
    if "search" in bundle
    and len(bundle) <= 3
    and not {"offline", "realtime"} <= bundle
}

assert all("search" in bundle for bundle in valid_bundles)
sorted([show(bundle) for bundle in valid_bundles], key=lambda x: (len(x), x))

[['search'],
 ['export', 'search'],
 ['offline', 'search'],
 ['realtime', 'search'],
 ['export', 'offline', 'search'],
 ['export', 'realtime', 'search']]

## Problem 15 — Inclusion–exclusion with verification

Implement the three-set inclusion–exclusion formula for union cardinality.

### Solution 15

In [41]:
def union_size_three(a: set[T], b: set[T], c: set[T]) -> int:
    return (
        len(a) + len(b) + len(c)
        - len(a & b) - len(a & c) - len(b & c)
        + len(a & b & c)
    )

a = set(range(0, 20, 2))
b = set(range(0, 20, 3))
c = set(range(0, 20, 5))

formula_size = union_size_three(a, b, c)
direct_size = len(a | b | c)

assert formula_size == direct_size
formula_size

14

## Problem 16 — Exact set equality after normalization

Normalize email addresses, compare unique sets, and report differences.

In [42]:
left_emails = [
    " Ada@example.com ",
    "bo@example.com",
    "ADA@example.com",
    "cy@example.com",
]
right_emails = [
    "ada@example.com",
    " cy@example.com ",
    "di@example.com",
]

### Solution 16

In [43]:
def normalize_email(email: str) -> str:
    return email.strip().casefold()

def compare_normalized(
    left: Iterable[str],
    right: Iterable[str],
) -> dict[str, object]:
    a = {normalize_email(value) for value in left}
    b = {normalize_email(value) for value in right}

    return {
        "equal": a == b,
        "only_left": a - b,
        "only_right": b - a,
        "symmetric_difference": a ^ b,
    }

comparison = compare_normalized(left_emails, right_emails)

assert comparison["only_left"] == {"bo@example.com"}
assert comparison["only_right"] == {"di@example.com"}

{k: show(v) if isinstance(v, set) else v for k, v in comparison.items()}

{'equal': False,
 'only_left': ['bo@example.com'],
 'only_right': ['di@example.com'],
 'symmetric_difference': ['bo@example.com', 'di@example.com']}

## Problem 17 — Streaming duplicate detector

Track seen hashable items with contains, add, bulk update, reset, and length operations.

### Solution 17

In [44]:
class SeenTracker:
    def __init__(self, initial: Iterable[T] = ()) -> None:
        self._seen: set[T] = set(initial)

    def contains(self, item: T) -> bool:
        return item in self._seen

    def add(self, item: T) -> bool:
        """Add item and return True only when it was new."""
        if item in self._seen:
            return False
        self._seen.add(item)
        return True

    def update(self, items: Iterable[T]) -> int:
        """Add many items and return the number of new distinct values."""
        before = len(self._seen)
        self._seen.update(items)
        return len(self._seen) - before

    def reset(self) -> None:
        self._seen.clear()

    def __len__(self) -> int:
        return len(self._seen)

tracker = SeenTracker(["A"])
events = ["A", "B", "C", "B", "D"]
new_flags = [tracker.add(event) for event in events]

assert new_flags == [False, True, True, False, True]
assert len(tracker) == 4
new_flags, len(tracker)

([False, True, True, False, True], 4)

## Problem 18 — Canonical cache keys with `frozenset`

Equivalent filter and option orderings must produce identical hashable cache keys.

### Solution 18

In [45]:
def cache_key(
    query_name: str,
    filters: Iterable[str],
    options: Mapping[str, Hashable],
) -> tuple[str, frozenset[str], frozenset[tuple[str, Hashable]]]:
    return (
        query_name,
        frozenset(filters),
        frozenset(options.items()),
    )

key_1 = cache_key(
    "search",
    ["python", "sets"],
    {"limit": 10, "exact": True},
)
key_2 = cache_key(
    "search",
    ["sets", "python", "sets"],
    {"exact": True, "limit": 10},
)

assert key_1 == key_2
hash(key_1), key_1

(-6748611163325892328,
 ('search',
  frozenset({'python', 'sets'}),
  frozenset({('exact', True), ('limit', 10)})))

## Problem 19 — Verify set identities exhaustively

Test algebraic identities over every subset of a small universe.

### Solution 19

In [46]:
universe = {0, 1, 2}
all_sets = [set(s) for s in powerset(universe)]

checks = 0
for A in all_sets:
    for B in all_sets:
        assert A | B == B | A
        assert A & B == B & A
        assert A ^ B == (A | B) - (A & B)

        complement_A = universe - A
        complement_B = universe - B
        assert universe - (A | B) == complement_A & complement_B
        assert universe - (A & B) == complement_A | complement_B

        for C in all_sets:
            assert A & (B | C) == (A & B) | (A & C)
            checks += 1

checks

512

## Problem 20 — Sparse support overlap

Represent sparse vectors with dictionaries and use their key sets as supports.

In [47]:
vector_a = {0: 2.0, 3: -1.0, 8: 4.0}
vector_b = {1: 7.0, 3: 5.0, 8: -2.0, 9: 1.0}

### Solution 20

In [48]:
def sparse_overlap_analysis(
    left: Mapping[int, float],
    right: Mapping[int, float],
) -> dict[str, object]:
    support_left = set(left)
    support_right = set(right)
    overlap = support_left & support_right

    dot_numerator = sum(left[i] * right[i] for i in overlap)

    return {
        "support_left": support_left,
        "support_right": support_right,
        "overlap": overlap,
        "only_left": support_left - support_right,
        "only_right": support_right - support_left,
        "dot_numerator": dot_numerator,
        "disjoint": support_left.isdisjoint(support_right),
    }

sparse_result = sparse_overlap_analysis(vector_a, vector_b)

assert sparse_result["overlap"] == {3, 8}
assert sparse_result["dot_numerator"] == -13.0

{
    key: show(value) if isinstance(value, set) else value
    for key, value in sparse_result.items()
}

{'support_left': [0, 3, 8],
 'support_right': [1, 3, 8, 9],
 'overlap': [3, 8],
 'only_left': [0],
 'only_right': [1, 9],
 'dot_numerator': -13.0,
 'disjoint': False}

# 4. Additional challenge extensions

1. **Weighted Jaccard:** compare dictionaries of nonnegative weights using sums of minima and maxima.
2. **Exact set cover:** search the powerset of candidates for the minimum cover and compare it with the greedy result.
3. **Graph cliques:** find all four-node cliques.
4. **Incremental inverted index:** add and remove documents without rebuilding the index.
5. **Constraint propagation:** repeatedly eliminate solved Sudoku values from peers.
6. **Access-control explanation:** report which roles contributed each permission.
7. **Dataset drift:** calculate added, removed, and stable categories.
8. **Pairwise-disjoint partition:** verify pairwise disjointness and exact union coverage.
9. **Dominating tags:** find tags occurring in at least 80% of items.
10. **Transitive dependency closure:** repeatedly union direct prerequisites.

# 5. Common pitfalls and best practices

## Pitfall 1 — Depending on output order

Set iteration order is not a sorting contract. Use `sorted(...)` for stable display and assertions.

## Pitfall 2 — Using `remove` when absence is acceptable

- `s.remove(x)` raises `KeyError` if `x` is absent.
- `s.discard(x)` is safe when absence is acceptable.

## Pitfall 3 — Assuming chained XOR means “exactly one”

For three or more sets, XOR means odd membership count. Use `Counter` for exact frequencies.

## Pitfall 4 — Mutating caller-owned inputs

Make a copy unless in-place behavior is documented.

## Pitfall 5 — Losing multiplicity

Sets discard duplicates. Use `Counter` when counts matter.

## Pitfall 6 — Using unhashable elements

Lists, dictionaries, and sets cannot be direct set elements. Use tuples or `frozenset` when appropriate.

## Pitfall 7 — Confusing partial ordering with numeric ordering

Two sets can be incomparable: neither a subset nor a superset of the other.

In [49]:
# Final regression suite.
assert jaccard([], []) == 1.0
assert jaccard([1, 2], [2, 3]) == 1 / 3
assert powerset(set()) == {frozenset()}
assert elements_with_frequency([{1, 2}, {2, 3}], 1) == {1, 3}
assert at_least_k([], 1) == set()
assert sudoku_candidates([], [], []) == set(range(1, 10))
assert cache_key("q", ["a", "b"], {"x": 1}) == cache_key(
    "q", ["b", "a"], {"x": 1}
)

print("All final regression checks passed.")

All final regression checks passed.


# 6. Summary reference

| Goal | Idiom |
|---|---|
| Common elements | `A & B` |
| All distinct elements | `A | B` |
| In left only | `A - B` |
| In exactly one of two | `A ^ B` |
| No overlap | `A.isdisjoint(B)` |
| Required values all present | `required <= actual` |
| Strict containment | `A < B` |
| Add many in place | `A.update(values)` |
| Remove many in place | `A.difference_update(values)` |
| Immutable/hashable set | `frozenset(values)` |
| Exact frequency across many sets | `Counter(...)` |
| Deterministic display | `sorted(A)` |

The key design question is: **What membership rule describes the desired result?**